# Portafolio de visualizaciones interpretadas — Delitos en Colombia (2020-2026)

**Actividad 15 híbrida — Representación y métodos gráficos**

**Enfoque social:** este portafolio compara delitos a nivel nacional para identificar líneas de tiempo (2020-2026) y diferencias geográficas entre departamentos y municipios de Colombia, usando datos oficiales de la Policía Nacional filtrados a seis delitos de alto impacto social: violencia intrafamiliar, amenazas, delitos sexuales, homicidio intencional, extorsión y secuestro.

Para cada pregunta analítica se sigue la estructura de interpretación:

> **Hallazgo:** patrón visible en el gráfico.
> **Evidencia:** cifras o elementos del gráfico que lo sustentan.
> **Precaución:** limitación o algo que el gráfico no permite concluir.


In [14]:
import sys
from pathlib import Path

RAIZ_PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.append(str(RAIZ_PROYECTO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingesta import cargar_consolidado
from src.graficos import aplicar_estilo

aplicar_estilo()
pd.options.display.float_format = "{:,.2f}".format

df = cargar_consolidado()
df.shape

AttributeError: 'ExcelFile' object has no attribute 'io'

## Diccionario de datos



## Pregunta 1 (Distribución): ¿cómo se distribuye la cantidad de casos registrados entre los departamentos de Colombia con mas casos (2020-2026)?

In [ ]:
casos_municipio = (
    df.groupby(["DEPARTAMENTO", "MUNICIPIO"])["CANTIDAD"]
      .sum()
      .reset_index(name="CASOS")
)

valores = casos_municipio["CASOS"]
bins = np.logspace(np.log10(valores.min()), np.log10(valores.max()), 25)

plt.figure(figsize=(9, 5))
plt.hist(valores, bins=bins, color="cornflowerblue", edgecolor="white")
plt.xscale("log")
plt.title("Distribución de casos registrados por municipio (2020-2026)")
plt.xlabel("Casos acumulados por municipio (escala logarítmica)")
plt.ylabel("Número de municipios")
plt.tight_layout()
plt.show()

casos_municipio["CASOS"].describe()

## Pregunta 2 (Comparativa): ¿cómo varía mes a mes la cantidad de casos entre los departamentos con más registros?

In [ ]:
top_departamentos = (
    df.groupby("DEPARTAMENTO")["CANTIDAD"].sum().sort_values(ascending=False).head(6).index.tolist()
)

mensual_depto = (
    df[df["DEPARTAMENTO"].isin(top_departamentos)]
      .groupby(["DEPARTAMENTO", "AÑO", "MES"])["CANTIDAD"]
      .sum()
      .reset_index(name="CASOS_MES")
)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=mensual_depto,
    x="DEPARTAMENTO",
    y="CASOS_MES",
    order=top_departamentos,
    hue="DEPARTAMENTO",
    legend=False,
)
plt.title("Variabilidad mensual de casos en los 6 departamentos con más registros (2020-2026)")
plt.xlabel("Departamento")
plt.ylabel("Casos por mes")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Pregunta 3 (Dispersión): ¿existe relación entre el número de casos de un departamento en 2020 y en 2025?

In [ ]:
por_anio = df.groupby(["DEPARTAMENTO", "AÑO"])["CANTIDAD"].sum().unstack(fill_value=0)
por_anio.columns = [str(c) for c in por_anio.columns]
por_anio = por_anio.reset_index()

correlacion = por_anio["2020"].corr(por_anio["2025"])

plt.figure(figsize=(8, 8))
sns.scatterplot(data=por_anio, x="2020", y="2025", s=80, alpha=0.8)

limite = max(por_anio["2020"].max(), por_anio["2025"].max()) * 1.05
plt.plot([0, limite], [0, limite], linestyle="--", color="gray", label="Mismo número de casos (2020 = 2025)")

destacados = por_anio[por_anio["DEPARTAMENTO"].isin(["BOGOTA", "ANTIOQUIA", "CÓRDOBA", "ATLÁNTICO"])]
for _, fila in destacados.iterrows():
    plt.annotate(fila["DEPARTAMENTO"], (fila["2020"], fila["2025"]), textcoords="offset points", xytext=(6, 6))

plt.title("Casos por departamento: 2020 vs. 2025 (años completos)")
plt.xlabel("Casos en 2020")
plt.ylabel("Casos en 2025")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Correlación de Pearson: {correlacion:.3f}")